In [3]:
import random
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta

import pandas as pd
from pytrends.request import TrendReq

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [4]:
# Parameters
START_DATE = "2018-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")
MONTH_STEP = 8  # 8 maanden om daggranulariteit stabiel te houden
SLEEP_BETWEEN_REQUESTS = 15
SLEEP_JITTER_SECONDS = 8
WINDOW_START_COOLDOWN = 30
MAX_RETRIES = 6
FILL_MISSING_WITH_ZERO = True

KEYWORDS = [
    "bitcoin", "cryptocurrency", "ethereum", "crypto", "btc",
    "eth", "crypto crash", "bitcoin crash", "buy bitcoin", "sell bitcoin",
    "crypto bull run", "solana", "altcoin", "bitcoin price", "crypto wallet",
]

pytrends = TrendReq(hl="en-US", tz=360)

In [ ]:
def chunk_keywords(keywords, chunk_size=5):
    """Google Trends accepteert max 5 termen per request."""
    return [keywords[i:i + chunk_size] for i in range(0, len(keywords), chunk_size)]


def generate_time_windows(start_date, end_date, months_step=8):
    """Maak vensters klein genoeg zodat we dagdata krijgen."""
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    windows = []
    while start < end:
        window_end = min(start + relativedelta(months=months_step), end)
        windows.append((start.date().isoformat(), window_end.date().isoformat()))
        start = window_end

    return windows


def polite_sleep(base_seconds=SLEEP_BETWEEN_REQUESTS, jitter_seconds=SLEEP_JITTER_SECONDS):
    """Slaap met jitter zodat requests niet in een herkenbaar patroon komen."""
    wait = base_seconds + random.uniform(0, jitter_seconds)
    time.sleep(wait)


def is_daily_granularity(df):
    if df.empty or len(df.index) < 3:
        return True
    steps = df.index.to_series().diff().dt.days.dropna()
    if steps.empty:
        return True
    return steps.mode().iloc[0] <= 1


def fetch_batch_trends(batch, start_date, end_date, retries=6):
    """Haal trends op voor een keyword-batch met retries + backoff."""
    cooldown = 20
    while retries > 0:
        try:
            pytrends.build_payload(batch, timeframe=f"{start_date} {end_date}")
            df = pytrends.interest_over_time()
            if "isPartial" in df.columns:
                df = df[df["isPartial"] == False]
            return df
        except Exception as exc:
            retries -= 1
            message = str(exc)
            is_rate_limit = "429" in message

            print(f"Fout voor batch {batch}: {message}. Resterende retries: {retries}")

            if is_rate_limit:
                wait_time = min(cooldown * 2, 240)
            else:
                wait_time = cooldown

            time.sleep(wait_time + random.uniform(0, 5))
            cooldown = min(cooldown * 2, 180)

    return pd.DataFrame()


def fetch_batch_trends_daily(batch, start_date, end_date):
    """
    Probeert eerst direct. Als Trends toch weekdata geeft,
    splitst deze functie automatisch in kleinere subvensters.
    """
    frame = fetch_batch_trends(batch, start_date, end_date, retries=MAX_RETRIES)
    if frame.empty:
        return frame

    if is_daily_granularity(frame):
        return frame

    print(f"Weekdata gedetecteerd voor {batch}. Splits naar 3-maands subvensters.")
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    pieces = []

    while start < end:
        sub_end = min(start + relativedelta(months=3), end)
        sub = fetch_batch_trends(
            batch,
            start.date().isoformat(),
            sub_end.date().isoformat(),
            retries=MAX_RETRIES,
        )
        if not sub.empty:
            pieces.append(sub)
        polite_sleep()
        start = sub_end

    if not pieces:
        return pd.DataFrame()

    out = pd.concat(pieces, axis=0)
    out = out.sort_index()
    out = out[~out.index.duplicated(keep="last")]
    return out


def fetch_trends_dataset(keywords, start_date, end_date, months_step=8):
    """
    - Batches per venster naast elkaar (axis=1)
    - Vensters onder elkaar (axis=0)
    """
    window_frames = []
    batches = chunk_keywords(keywords, 5)
    windows = generate_time_windows(start_date, end_date, months_step)

    for window_start, window_end in windows:
        print(f"Periode: {window_start} -> {window_end}")
        # Extra rust aan begin van ieder venster voorkomt vaak 429 op eerste batch
        time.sleep(WINDOW_START_COOLDOWN)

        batch_frames = []
        for batch in batches:
            frame = fetch_batch_trends_daily(batch, window_start, window_end)
            if not frame.empty:
                batch_frames.append(frame)
            polite_sleep()

        if batch_frames:
            merged_window = pd.concat(batch_frames, axis=1)
            merged_window = merged_window.loc[:, ~merged_window.columns.duplicated(keep="last")]
            merged_window = merged_window.reindex(columns=keywords + ["isPartial"], fill_value=pd.NA)
            window_frames.append(merged_window)

    if not window_frames:
        return pd.DataFrame()

    combined = pd.concat(window_frames, axis=0)
    combined = combined.sort_index()
    combined = combined[~combined.index.duplicated(keep="last")]
    return combined

In [6]:
# Data ophalen
df_raw = fetch_trends_dataset(
    keywords=KEYWORDS,
    start_date=START_DATE,
    end_date=END_DATE,
    months_step=MONTH_STEP,
)

print(f"Aantal rijen opgehaald: {len(df_raw)}")
df_raw.head()

Periode: 2018-01-01 -> 2018-09-01
Periode: 2018-09-01 -> 2019-05-01
Periode: 2019-05-01 -> 2020-01-01
Periode: 2020-01-01 -> 2020-09-01
Periode: 2020-09-01 -> 2021-05-01
Periode: 2021-05-01 -> 2022-01-01
Periode: 2022-01-01 -> 2022-09-01
Periode: 2022-09-01 -> 2023-05-01
Periode: 2023-05-01 -> 2024-01-01
Periode: 2024-01-01 -> 2024-09-01
Periode: 2024-09-01 -> 2025-05-01
Periode: 2025-05-01 -> 2026-01-01
Periode: 2026-01-01 -> 2026-04-15


/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
/Users/mrezeddin/.pyenv/versions/3.12.12/lib/python3.12/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.in

Aantal rijen opgehaald: 3026


,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet,isPartial
date,,,,,,,,,,,,,,,,
2018-01-01,46,8,4,6,9,46,0,2,34,4,0,1,2,39,1,False
2018-01-02,53,10,8,8,12,69,0,3,42,6,0,1,2,48,1,False
2018-01-03,53,12,8,10,12,73,1,3,51,7,0,1,3,46,1,False
2018-01-04,52,14,9,11,13,92,1,3,55,6,0,1,3,41,2,False
2018-01-05,48,13,8,11,13,82,1,3,51,7,0,1,4,40,2,False


In [ ]:
if df_raw.empty:
    raise ValueError("Geen data opgehaald. Controleer connectie, keywords of API limits.")

print("Kolommen:", df_raw.columns.tolist())
print("Duplicaten op index:", int(df_raw.index.duplicated().sum()))
print("Aantal missende waarden:")
print(df_raw.isna().sum().sort_values(ascending=False).head(10))

Kolommen: ['bitcoin', 'cryptocurrency', 'ethereum', 'crypto', 'btc', 'eth', 'crypto crash', 'bitcoin crash', 'buy bitcoin', 'sell bitcoin', 'crypto bull run', 'solana', 'altcoin', 'bitcoin price', 'crypto wallet', 'isPartial']
Duplicaten op index: 0
Aantal missende waarden:
bitcoin           0
cryptocurrency    0
ethereum          0
crypto            0
btc               0
eth               0
crypto crash      0
bitcoin crash     0
buy bitcoin       0
sell bitcoin      0
dtype: int64


In [ ]:
if "isPartial" in df_raw.columns:
    df_clean = df_raw.drop(columns=["isPartial"]).copy()
else:
    df_clean = df_raw.copy()

df_clean.index = pd.to_datetime(df_clean.index, errors="coerce")
df_clean = df_clean[~df_clean.index.isna()]

df_clean = df_clean.sort_index()
df_clean = df_clean[~df_clean.index.duplicated(keep="last")]

for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

day_steps = df_clean.index.to_series().diff().dt.days.dropna()
print("Meest voorkomende stap (dagen):")
print(day_steps.value_counts().head())

print("NaN per kolom VOOR imputatie:")
print(df_clean.isna().sum().sort_values(ascending=False).head(10))

# Optioneel imputeren voor modeltraining
if FILL_MISSING_WITH_ZERO:
    df_clean = df_clean.fillna(0)

print("Shape na cleaning:", df_clean.shape)
print("NaN totaal na cleaning:", int(df_clean.isna().sum().sum()))
df_clean.head()

Meest voorkomende stap (dagen):
date
1.0    3025
Name: count, dtype: int64
NaN per kolom VOOR imputatie:
bitcoin           0
cryptocurrency    0
ethereum          0
crypto            0
btc               0
eth               0
crypto crash      0
bitcoin crash     0
buy bitcoin       0
sell bitcoin      0
dtype: int64
Shape na cleaning: (3026, 15)
NaN totaal na cleaning: 0


,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet
date,,,,,,,,,,,,,,,
2018-01-01,46,8,4,6,9,46,0,2,34,4,0,1,2,39,1
2018-01-02,53,10,8,8,12,69,0,3,42,6,0,1,2,48,1
2018-01-03,53,12,8,10,12,73,1,3,51,7,0,1,3,46,1
2018-01-04,52,14,9,11,13,92,1,3,55,6,0,1,3,41,2
2018-01-05,48,13,8,11,13,82,1,3,51,7,0,1,4,40,2


In [9]:
df_clean.tail()

,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet
date,,,,,,,,,,,,,,,
2026-04-10,19,1,2,9,8,31,1,1,7,1,0,5,0,14,1
2026-04-11,17,1,2,10,7,28,1,1,7,1,0,4,0,12,1
2026-04-12,18,1,2,9,8,28,0,1,6,1,0,4,0,14,1
2026-04-13,19,1,2,10,8,33,0,0,7,1,0,5,0,16,1
2026-04-14,21,1,3,9,8,37,0,0,6,1,0,5,0,16,1


In [10]:
# Finale validatie + export
print("Datum bereik:", df_clean.index.min(), "tot", df_clean.index.max())
print("Totaal missende waarden:", int(df_clean.isna().sum().sum()))

output_path = "google_trends_clean.csv"
df_clean.to_csv(output_path, index=True)
print(f"Opgeslagen als: {output_path}")

Datum bereik: 2018-01-01 00:00:00 tot 2026-04-14 00:00:00
Totaal missende waarden: 0
Opgeslagen als: google_trends_clean.csv
